# Prétraitement des données

## Réalisé Par :

- [Samain Florian](https://github.com/NwaSet) — 2ème année, Intar D  
- [Ducourtieux Yohann](https://github.com/Nhanyo) — 2ème année, Intar D  

## Objectif :

Dans cette section, nous passons à l’étape de prétraitement de notre projet. Cette étape est essentielle dans tout workflow d’analyse de données ou de machine learning, car elle consiste à préparer les données brutes avant de construire un modèle prédictif.

Les données issues du monde réel sont souvent incomplètes, incohérentes ou bruitées. Le prétraitement vise donc à nettoyer et transformer ces données afin d’obtenir un format structuré et fiable. Cela inclut la gestion des valeurs manquantes, la correction des incohérences, la suppression des valeurs aberrantes, l’encodage des variables catégorielles ainsi que la mise à l’échelle des variables numériques lorsque cela est nécessaire.

En réalisant ces opérations, nous nous assurons que le jeu de données est adapté à la modélisation et que les algorithmes peuvent apprendre efficacement. Un bon prétraitement permet d’améliorer les performances du modèle, de réduire les biais potentiels et d’obtenir des résultats plus précis et pertinents.

## Table des matières

- [Réalisé par](#réalisé-Par-:)
- [Objectif](#Objectif)
- [Importation Des Librairie + Fonction Utile](#Importation-Des-Librairie-+-Fonction-Utile-:)

## Importation Des Librairie + Fonction Utile :

L’objectif de cette partie est d’importer les bibliothèques nécessaires au prétraitement des données, ainsi que de créer des fonctions réutilisables que nous pourrons utiliser à plusieurs reprises.

In [71]:
# Manipulation et gestion des données
import pandas as pd
import numpy as np

# Visualisation des données
import matplotlib.pyplot as plt

# Séparation du dataset en ensembles d'entraînement et de test
from sklearn.model_selection import train_test_split

# Encodage des variables catégorielles
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import OneHotEncoder

# Normalisation / standardisation des variables numériques 
from sklearn.preprocessing import StandardScaler

In [72]:
def list_to_string(liste):
    """
    Convertit une liste en chaîne de caractères séparée par des virgules.
    
    param liste: list - La liste à convertir
    return: str - La chaîne résultante
    """
    if not isinstance(liste, list):
        raise TypeError("Le paramètre doit être une liste.")
    
    # Conversion de chaque élément en chaîne pour éviter les erreurs
    return ",".join(str(element) for element in liste)

# Initialisation Des Données 

Dans un premier temps, l’objectif est de créer un DataFrame à l’aide de pandas. Nous séparons ensuite la variable cible des variables explicatives, puis nous divisons les données en ensembles d’entraînement (80%) et de test (20%).
- `y_titanic` contient uniquement la cible.
- `X_titanic` contient toutes les autres informations.

In [73]:
df_titanic = pd.read_csv('../data/Titanic Dataset.csv')

y_titanic = df_titanic['survived'].to_numpy()

df_titanic = df_titanic.drop(['survived'], axis=1)
column_names_X=df_titanic.keys().to_list()
X_titanic = df_titanic.to_numpy()

print(y_titanic.shape)
print(X_titanic.shape)


X_train, X_test, y_train, y_test = train_test_split(X_titanic, y_titanic, test_size=0.2, stratify=y_titanic, random_state=2)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1309,)
(1309, 10)
(1047, 10)
(262, 10)
(1047,)
(262,)


## séparation par type de variable 
l'objectif ici est de séparer les types de données pour pouvoir les travailler plus facilement. on ne touchera pas a notre target car elle a déjà un format correct pour l'apprentissage. 

In [74]:
# binaire
X_test_binary = X_test[:, [2]]
X_train_binary = X_train[:, [2]]

# catégoriel
X_test_categ = X_test[:, [0, 8, 9]]
X_train_categ = X_train[:, [0, 8, 9]]

# numérique
X_test_num = X_test[:, [3, 4, 5, 7]]
X_train_num = X_train[:, [3, 4, 5, 7]]

## traitement des variables binaires

In [75]:
label = LabelBinarizer()

X_test_binary = label.fit_transform(X_test_binary)
X_train_binary = label.fit_transform(X_train_binary)

# print(X_test_binary) # -----> work

## traitement des variables catégoriel

In [76]:
one_hot = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

X_train_categ = one_hot.fit_transform(X_train_categ)
X_test_categ = one_hot.transform(X_test_categ)

print(X_test_categ)

[[0. 1. 0. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 1. 0.]
 ...
 [0. 1. 0. ... 0. 1. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]]


## Normalisation des variables numériques

In [77]:
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

print(X_test_num)

[[-0.41451452  0.46971064 -0.44332755 -0.13329134]
 [-0.13770682  1.40466698 -0.44332755 -0.50565483]
 [-0.62212029 -0.46524571 -0.44332755 -0.50565483]
 ...
 [ 2.07675478  0.46971064  0.8010798   0.13452197]
 [ 1.38473553 -0.46524571 -0.44332755 -0.07741106]
 [-1.45254339  0.46971064  0.8010798  -0.35483886]]


## Concaténation des données

In [78]:
X_train_final = np.concatenate(
    [X_train_binary, X_train_categ, X_train_num],
    axis=1
)

X_test_final = np.concatenate(
    [X_test_binary, X_test_categ, X_test_num],
    axis=1
)

print("X_train_final :", X_train_final.shape)
print("X_test_final  :", X_test_final.shape)
print("y_train       :", y_train.shape)
print("y_test        :", y_test.shape)

X_train_final : (1047, 173)
X_test_final  : (262, 173)
y_train       : (1047,)
y_test        : (262,)


In [79]:
# =========================
# Création des noms de colonnes
# =========================

# 1. Binaire
binary_cols = ["Sex"]  # adapte si nécessaire

# 2. Catégorielles (important)
categ_original_cols = [
    column_names_X[0],
    column_names_X[8],
    column_names_X[9]
]

categ_cols = one_hot.get_feature_names_out(categ_original_cols)

# 3. Numériques
num_cols = [
    column_names_X[3],
    column_names_X[4],
    column_names_X[5],
    column_names_X[7]
]

# 4. Fusion de tous les noms
all_columns = list(binary_cols) + list(categ_cols) + list(num_cols)

In [80]:
df_X_train = pd.DataFrame(X_train_final, columns=all_columns)
df_X_test = pd.DataFrame(X_test_final, columns=all_columns)

## Exportation des données

In [81]:
export_path = "../data/pre_processed"

df_X_train.to_csv(f"{export_path}/X_train_preprocessed.csv", index=False)
df_X_test.to_csv(f"{export_path}/X_test_preprocessed.csv", index=False)
pd.DataFrame(y_train, columns=["survived"]).to_csv(f"{export_path}/y_train.csv", index=False)
pd.DataFrame(y_test, columns=["survived"]).to_csv(f"{export_path}/y_test.csv", index=False)

